# GDS (Graph Data Science) 예제

이 노트북에서는 Neo4j의 Graph Data Science 라이브러리를 사용하여 그래프 분석을 수행하는 방법을 학습합니다.

## 학습 목표
- GDS 라이브러리 기본 사용법
- 그래프 프로젝션 생성
- PageRank 알고리즘 실행
- Community Detection (커뮤니티 탐지)
- 노드 유사도 계산
- 결과 시각화 및 활용


## (옵션) Neo4j 설치


### 설치 방법

1. docker-compose.yml 파일이 있는 폴더로 이동
2. `docker-compose up -d` 명령어 실행

**참고**: GDS는 Neo4j Enterprise 버전에서 기본 제공되거나, Community 버전에서는 플러그인으로 설치해야 합니다.


### WEB UI 접속

1. `http://localhost:7474` 접속
2. 아이디 / 비번 : `neo4j/test1234`


## 1. GDS란?

**Graph Data Science (GDS)**는 Neo4j에서 제공하는 그래프 데이터 과학 라이브러리입니다.

### 주요 기능
- **그래프 알고리즘**: PageRank, Centrality, Community Detection 등
- **그래프 임베딩**: 노드를 벡터로 변환
- **그래프 네이티브 ML**: 그래프 구조를 활용한 머신러닝
- **성능 최적화**: 대규모 그래프 데이터 효율적 처리

### 활용 분야
- 추천 시스템
- 소셜 네트워크 분석
- 사기 탐지
- GraphRAG
- 지식 그래프 분석


## 2. 라이브러리 설치 및 Import


In [ ]:
# 필요한 라이브러리 import
from graphdatascience import GraphDataScience
import pandas as pd

# ============================================
# Neo4j 연결 정보 설정
# ============================================
URI = "bolt://localhost:7687"
USERNAME = "neo4j"
PASSWORD = "test1234"

# ============================================
# GDS 클라이언트 생성
# ============================================
# GraphDataScience: GDS 라이브러리의 메인 클래스
# Neo4j 드라이버를 자동으로 생성하고 관리합니다.
gds = GraphDataScience(URI, auth=(USERNAME, PASSWORD))

print("GDS 클라이언트가 생성되었습니다!")
print(f"GDS 버전: {gds.version()}")


## 3. 샘플 데이터 생성

GDS 알고리즘을 테스트하기 위한 샘플 그래프 데이터를 생성합니다.


In [ ]:
# ============================================
# 기존 데이터 삭제 (깨끗한 상태로 시작)
# ============================================
gds.run_cypher("""
MATCH (n)
DETACH DELETE n
""")
print("기존 데이터를 삭제했습니다.")

# ============================================
# 샘플 그래프 데이터 생성
# ============================================
# 소셜 네트워크를 시뮬레이션하는 그래프 생성
# - Person 노드들 간의 KNOWS 관계
# - 다양한 연결 패턴으로 알고리즘 테스트 가능
gds.run_cypher("""
// 그룹 A: 홍길동과 연결된 사람들
CREATE 
    (hong:Person {name: '홍길동', group: 'A'}),
    (shin:Person {name: '신사임당', group: 'A'}),
    (lee:Person {name: '이순신', group: 'A'}),
    (sejong:Person {name: '세종대왕', group: 'A'}),
    
// 그룹 B: 김철수와 연결된 사람들
    (kim:Person {name: '김철수', group: 'B'}),
    (park:Person {name: '박영희', group: 'B'}),
    (choi:Person {name: '최민수', group: 'B'}),
    
// 그룹 C: 독립적인 그룹
    (jung:Person {name: '정수진', group: 'C'}),
    (yoon:Person {name: '윤지훈', group: 'C'}),

// 그룹 A 내부 연결
    (hong)-[:KNOWS {weight: 1.0}]->(shin),
    (hong)-[:KNOWS {weight: 0.8}]->(lee),
    (hong)-[:KNOWS {weight: 0.9}]->(sejong),
    (shin)-[:KNOWS {weight: 1.0}]->(lee),
    (lee)-[:KNOWS {weight: 0.7}]->(sejong),

// 그룹 B 내부 연결
    (kim)-[:KNOWS {weight: 1.0}]->(park),
    (kim)-[:KNOWS {weight: 0.9}]->(choi),
    (park)-[:KNOWS {weight: 0.8}]->(choi),

// 그룹 C 내부 연결
    (jung)-[:KNOWS {weight: 1.0}]->(yoon),

// 그룹 간 연결 (약한 연결)
    (hong)-[:KNOWS {weight: 0.3}]->(kim),
    (lee)-[:KNOWS {weight: 0.2}]->(park)
""")

print("샘플 그래프 데이터가 생성되었습니다!")


In [ ]:
# 생성된 데이터 확인
result = gds.run_cypher("""
MATCH (p:Person)
RETURN p.name as 이름, p.group as 그룹, 
       size((p)-[:KNOWS]->()) as 연결수
ORDER BY 연결수 DESC
""")

print("생성된 노드 정보:")
for record in result:
    print(f"  - {record['이름']} (그룹: {record['그룹']}, 연결수: {record['연결수']})")


## 4. 그래프 프로젝션 (Graph Projection)

GDS 알고리즘을 실행하기 전에 Neo4j의 그래프 데이터를 GDS 형식으로 프로젝션해야 합니다.

### 프로젝션이 필요한 이유
- GDS는 최적화된 인메모리 그래프 구조를 사용
- 알고리즘 실행 성능 향상
- 대규모 그래프 데이터 처리 가능


In [ ]:
# ============================================
# 그래프 프로젝션 생성
# ============================================
# gds.graph.project(): Neo4j 그래프를 GDS 형식으로 프로젝션
#   - "myGraph": 프로젝션 이름
#   - "Person": 노드 레이블
#   - "KNOWS": 관계 타입
#   - nodeProperties: 노드 속성 (알고리즘에서 사용 가능)
#   - relationshipProperties: 관계 속성 (가중치 등)

G, result = gds.graph.project(
    "myGraph",                    # 프로젝션 이름
    "Person",                     # 노드 레이블
    "KNOWS",                      # 관계 타입
    nodeProperties=["name", "group"],  # 노드 속성
    relationshipProperties="weight"   # 관계 속성 (가중치)
)

print(f"그래프 프로젝션이 생성되었습니다!")
print(f"  - 노드 수: {result['nodeCount']}")
print(f"  - 관계 수: {result['relationshipCount']}")
print(f"  - 프로젝션 이름: {G.name()}")


## 5. PageRank 알고리즘

**PageRank**는 노드의 중요도(영향력)를 계산하는 알고리즘입니다. 
- 연결이 많고, 중요한 노드와 연결된 노드일수록 높은 점수를 받습니다.
- 검색 엔진, 소셜 네트워크 분석, 추천 시스템 등에 활용됩니다.


In [ ]:
# ============================================
# PageRank 알고리즘 실행
# ============================================
# gds.pageRank.stream(): PageRank 점수를 계산하고 스트림으로 반환
#   - G: 프로젝션된 그래프
#   - maxIterations: 최대 반복 횟수 (기본값: 20)
#   - dampingFactor: 댐핑 팩터 (기본값: 0.85)
#     - 0.85는 일반적으로 사용되는 값 (Google의 원래 값)
#     - 높을수록 더 많은 확률이 다른 노드로 전파됨

pagerank_result = gds.pageRank.stream(G, maxIterations=20, dampingFactor=0.85)

# 결과를 DataFrame으로 변환하여 보기 쉽게 출력
df_pagerank = pagerank_result.sort_values('score', ascending=False)

print("PageRank 점수 (높은 순):")
print("=" * 50)
for idx, row in df_pagerank.iterrows():
    print(f"{row['nodeId']:2d}번 노드: {row['score']:.4f}")

print("\n노드 이름과 함께 확인:")
# nodeId를 실제 노드 이름으로 매핑
node_info = gds.run_cypher("""
MATCH (p:Person)
RETURN id(p) as nodeId, p.name as name
""")

# DataFrame 병합
df_pagerank_with_name = df_pagerank.merge(node_info, on='nodeId', how='left')
df_pagerank_with_name = df_pagerank_with_name.sort_values('score', ascending=False)

for idx, row in df_pagerank_with_name.iterrows():
    print(f"  {row['name']:8s}: {row['score']:.4f}")


In [ ]:
# ============================================
# PageRank 결과를 노드 속성으로 저장
# ============================================
# gds.pageRank.write(): PageRank 점수를 계산하고 노드 속성으로 저장
#   - writeProperty: 저장할 속성 이름
#   - 이후 Cypher 쿼리에서 이 속성을 사용할 수 있음

pagerank_write_result = gds.pageRank.write(
    G,
    maxIterations=20,
    dampingFactor=0.85,
    writeProperty="pagerank"  # 노드에 저장될 속성 이름
)

print(f"PageRank 점수가 노드 속성으로 저장되었습니다!")
print(f"  - 처리된 노드 수: {pagerank_write_result['nodePropertiesWritten']}")

# 저장된 결과 확인
result = gds.run_cypher("""
MATCH (p:Person)
RETURN p.name as 이름, p.pagerank as PageRank점수
ORDER BY p.pagerank DESC
""")

print("\n저장된 PageRank 점수:")
for record in result:
    print(f"  {record['이름']:8s}: {record['PageRank점수']:.4f}")


## 6. Community Detection (커뮤니티 탐지)

**커뮤니티 탐지**는 그래프에서 밀집하게 연결된 노드 그룹을 찾는 알고리즘입니다.
- **Louvain**: 가장 널리 사용되는 커뮤니티 탐지 알고리즘
- **Leiden**: Louvain의 개선 버전
- 소셜 네트워크에서 친구 그룹, 지식 그래프에서 주제별 클러스터 등을 찾을 때 활용


In [ ]:
# ============================================
# Louvain 커뮤니티 탐지 알고리즘 실행
# ============================================
# gds.louvain.stream(): Louvain 알고리즘으로 커뮤니티를 찾고 스트림으로 반환
#   - communityId: 각 노드가 속한 커뮤니티 ID
#   - 같은 ID를 가진 노드들이 하나의 커뮤니티

louvain_result = gds.louvain.stream(G)

# 결과를 DataFrame으로 변환
df_louvain = louvain_result.sort_values('communityId')

print("Louvain 커뮤니티 탐지 결과:")
print("=" * 50)

# 커뮤니티별로 그룹화
communities = {}
for idx, row in df_louvain.iterrows():
    comm_id = row['communityId']
    if comm_id not in communities:
        communities[comm_id] = []
    communities[comm_id].append(row['nodeId'])

# 노드 이름과 함께 출력
node_info = gds.run_cypher("""
MATCH (p:Person)
RETURN id(p) as nodeId, p.name as name, p.group as group
""")

df_louvain_with_name = df_louvain.merge(node_info, on='nodeId', how='left')

for comm_id in sorted(communities.keys()):
    members = df_louvain_with_name[df_louvain_with_name['communityId'] == comm_id]
    print(f"\n커뮤니티 {comm_id}:")
    for idx, row in members.iterrows():
        print(f"  - {row['name']} (원래 그룹: {row['group']})")


In [ ]:
# ============================================
# 커뮤니티 ID를 노드 속성으로 저장
# ============================================
louvain_write_result = gds.louvain.write(
    G,
    writeProperty="community"  # 노드에 저장될 속성 이름
)

print(f"커뮤니티 ID가 노드 속성으로 저장되었습니다!")
print(f"  - 발견된 커뮤니티 수: {louvain_write_result['communityCount']}")
print(f"  - 처리된 노드 수: {louvain_write_result['nodePropertiesWritten']}")

# 저장된 결과 확인
result = gds.run_cypher("""
MATCH (p:Person)
RETURN p.community as 커뮤니티ID, 
       collect(p.name) as 멤버들,
       collect(p.group) as 원래그룹
ORDER BY 커뮤니티ID
""")

print("\n커뮤니티별 멤버:")
for record in result:
    print(f"\n커뮤니티 {record['커뮤니티ID']}:")
    for name in record['멤버들']:
        print(f"  - {name}")


## 7. 노드 유사도 (Node Similarity)

**노드 유사도**는 두 노드가 얼마나 유사한 이웃을 가지고 있는지 계산합니다.
- 공통 이웃이 많을수록 높은 유사도
- 추천 시스템에서 "이 노드와 유사한 노드"를 찾을 때 활용


In [ ]:
# ============================================
# 노드 유사도 계산
# ============================================
# gds.nodeSimilarity.stream(): 노드 간 유사도를 계산하고 스트림으로 반환
#   - similarityCutoff: 유사도 임계값 (이 값 이상인 관계만 반환)
#   - topK: 각 노드당 상위 K개의 유사한 노드만 반환

similarity_result = gds.nodeSimilarity.stream(
    G,
    similarityCutoff=0.1,  # 유사도 0.1 이상인 관계만
    topK=5  # 각 노드당 상위 5개만
)

# 결과를 DataFrame으로 변환
df_similarity = similarity_result.sort_values('similarity', ascending=False)

print("노드 유사도 결과 (상위 10개):")
print("=" * 60)

# nodeId를 실제 노드 이름으로 매핑
node_info = gds.run_cypher("""
MATCH (p:Person)
RETURN id(p) as nodeId, p.name as name
""")

df_similarity_with_name = df_similarity.merge(
    node_info, 
    left_on='node1', 
    right_on='nodeId', 
    how='left'
)
df_similarity_with_name = df_similarity_with_name.rename(columns={'name': 'node1_name'})

df_similarity_with_name = df_similarity_with_name.merge(
    node_info, 
    left_on='node2', 
    right_on='nodeId', 
    how='left'
)
df_similarity_with_name = df_similarity_with_name.rename(columns={'name': 'node2_name'})

df_similarity_with_name = df_similarity_with_name.sort_values('similarity', ascending=False)

for idx, row in df_similarity_with_name.head(10).iterrows():
    print(f"  {row['node1_name']:8s} ↔ {row['node2_name']:8s}: {row['similarity']:.4f}")


## 8. 가중치 기반 알고리즘

관계에 가중치(weight) 속성이 있는 경우, 이를 활용하여 더 정확한 분석이 가능합니다.


In [ ]:
# ============================================
# 가중치 기반 PageRank
# ============================================
# relationshipWeightProperty: 관계의 가중치 속성 지정
#   - 가중치가 높은 관계를 더 중요하게 고려

pagerank_weighted_result = gds.pageRank.stream(
    G,
    maxIterations=20,
    dampingFactor=0.85,
    relationshipWeightProperty="weight"  # 관계의 weight 속성 사용
)

# 결과를 DataFrame으로 변환
df_pagerank_weighted = pagerank_weighted_result.sort_values('score', ascending=False)

print("가중치 기반 PageRank 점수:")
print("=" * 50)

# 노드 이름과 함께 출력
node_info = gds.run_cypher("""
MATCH (p:Person)
RETURN id(p) as nodeId, p.name as name
""")

df_pagerank_weighted_with_name = df_pagerank_weighted.merge(node_info, on='nodeId', how='left')

for idx, row in df_pagerank_weighted_with_name.iterrows():
    print(f"  {row['name']:8s}: {row['score']:.4f}")

print("\n비교: 일반 PageRank vs 가중치 기반 PageRank")
print("-" * 50)

# 일반 PageRank와 비교
df_comparison = df_pagerank_with_name.merge(
    df_pagerank_weighted_with_name[['nodeId', 'score']], 
    on='nodeId', 
    suffixes=('_일반', '_가중치')
)

for idx, row in df_comparison.iterrows():
    print(f"{row['name']:8s}: 일반={row['score_일반']:.4f}, 가중치={row['score_가중치']:.4f}")


## 9. 그래프 프로젝션 삭제

작업이 끝나면 그래프 프로젝션을 삭제하여 메모리를 확보할 수 있습니다.


In [ ]:
# ============================================
# 그래프 프로젝션 삭제
# ============================================
# gds.graph.drop(): 그래프 프로젝션을 삭제하여 메모리 확보
#   - 프로젝션은 인메모리 구조이므로 명시적으로 삭제해야 함

G.drop()
print("그래프 프로젝션이 삭제되었습니다.")


## 10. 정리 및 추가 학습

### 주요 GDS 알고리즘 요약

| 알고리즘 | 용도 | 주요 파라미터 |
|---------|------|--------------|
| **PageRank** | 노드 중요도 계산 | maxIterations, dampingFactor |
| **Louvain** | 커뮤니티 탐지 | - |
| **Node Similarity** | 노드 유사도 계산 | similarityCutoff, topK |
| **Betweenness Centrality** | 중개자 역할 노드 찾기 | - |
| **Shortest Path** | 최단 경로 계산 | - |

### GDS 사용 흐름
1. **그래프 프로젝션 생성**: `gds.graph.project()`
2. **알고리즘 실행**: `gds.[알고리즘].stream()` 또는 `.write()`
3. **결과 활용**: 노드 속성으로 저장하거나 분석
4. **프로젝션 삭제**: `G.drop()` (메모리 관리)


### 참고 자료
- [Neo4j GDS 공식 문서](https://neo4j.com/docs/graph-data-science/)
- [GDS 알고리즘 가이드](https://neo4j.com/docs/graph-data-science/current/algorithms/)
- [Python graphdatascience 라이브러리](https://neo4j.com/docs/graph-data-science-client/python/)
